In [20]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ast

df = pd.read_csv("./data/processed/indian_films_cleaned.csv")
df["release_date"] = pd.to_datetime(df["release_date"])
df["year"] = df["release_date"].dt.year

bo = df[df["has_boxoffice"]].copy()

# Use INR crore columns
bo["rev_cr"]    = bo["revenue_inr_crore"]
bo["budget_cr"] = bo["budget_inr_crore"]

SCORE_LABELS = {
    0: "Score 0 — No representation",
    2: "Score 2 — Meaningful presence",
    3: "Score 3 — Female-led + directed"
}
SCORE_COLORS = {0: "#e63946", 2: "#2a9d8f", 3: "#457b9d"}
print(f"✓ Loaded: {len(df)} films, {len(bo)} with box office data")
print(f"\nRevenue range: ₹{bo['rev_cr'].min():.1f} cr — ₹{bo['rev_cr'].max():.1f} cr")
print(f"\nTop 5:\n{bo.nlargest(5,'rev_cr')[['title','rev_cr','bechdel_proxy_score']].to_string(index=False)}")

✓ Loaded: 299 films, 197 with box office data

Revenue range: ₹2.1 cr — ₹2023.7 cr

Top 5:
                     title  rev_cr  bechdel_proxy_score
                    Dangal 2023.71                  2.0
Bāhubali 2: The Conclusion 1810.00                  0.0
       Pushpa 2 - The Rule 1755.40                  0.0
                       RRR 1200.00                  2.0
                     Jawan 1148.00                  2.0


In [22]:
# PLOT 2: Revenue boxplot — change revenue_m to rev_cr
for score, color in SCORE_COLORS.items():
    subset = bo[bo["bechdel_proxy_score"] == score]["rev_cr"].dropna()
    fig.add_trace(go.Box(
        y=subset,
        name=SCORE_LABELS[score],
        marker_color=color,
        boxmean=True,
        legendgroup=f"score_{score}",
        showlegend=False
    ), row=1, col=2)

# PLOT 4: Budget vs Revenue scatter — change budget_m/revenue_m to budget_cr/rev_cr
bo["marker_size"] = 6 + bo["female_lead_ratio"] * 20

for score, color in SCORE_COLORS.items():
    sub = bo[bo["bechdel_proxy_score"] == score]
    fig.add_trace(go.Scatter(
        x=sub["budget_cr"], y=sub["rev_cr"],
        mode="markers",
        marker=dict(color=color, size=sub["marker_size"], opacity=0.7,
                    line=dict(width=0.5, color="#333")),
        name=SCORE_LABELS[score],
        legendgroup=f"score_{score}",
        showlegend=False,
        text=sub["title"],
        customdata=np.stack([
            sub["female_lead_ratio"].round(2),
            sub["bechdel_proxy_score"],
            sub["year"]
        ], axis=-1),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Budget: ₹%{x:.0f} cr<br>"
            "Revenue: ₹%{y:.0f} cr<br>"
            "Female lead ratio: %{customdata[0]}<br>"
            "Year: %{customdata[2]}<extra></extra>"
        )
    ), row=2, col=2)

# Break-even line
max_val = bo["budget_cr"].max()
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode="lines", line=dict(color="#666", dash="dash", width=1),
    name="Break-even", showlegend=True
), row=2, col=2)

# PLOT 6: Top 15 — change revenue_m to rev_cr
top15 = bo.nlargest(15, "rev_cr")[["title", "rev_cr", "bechdel_proxy_score", "year"]].copy()
top15 = top15.sort_values("rev_cr")
top15["color"] = top15["bechdel_proxy_score"].map(SCORE_COLORS)

fig.add_trace(go.Bar(
    x=top15["rev_cr"],
    y=top15["title"],
    orientation="h",
    marker_color=top15["color"],
    text=top15["rev_cr"].apply(lambda x: f"₹{x:.0f} cr"),
    textposition="auto",
    customdata=top15[["bechdel_proxy_score", "year"]],
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Revenue: ₹%{x:.0f} cr<br>"
        "Year: %{customdata[1]}<br>"
        "Score: %{customdata[0]}<extra></extra>"
    ),
    showlegend=False
), row=3, col=2)

fig.update_layout(
    height=1400,
    width=1400,
    title=dict(
        text="<b>Gender Representation in Indian Cinema (2000–2024)</b><br>"
             "<sup>Bechdel Proxy Analysis — Hindi, Tamil & Telugu Films</sup>",
        x=0.5, xanchor="center",
        font=dict(size=18, color="white")
    ),
    paper_bgcolor="#0f0f0f",
    plot_bgcolor="#1a1a1a",
    font=dict(color="white", family="Inter, Arial, sans-serif", size=11),
    barmode="group",
    legend=dict(
        bgcolor="#1a1a1a", bordercolor="#444", borderwidth=1,
        font=dict(color="white", size=10),
        orientation="v",
        x=1.01, y=0.99,
        xanchor="left", yanchor="top",
        tracegroupgap=4
    ),
    margin=dict(l=80, r=250, t=140, b=80)
)

axis_style = dict(gridcolor="#2a2a2a", zerolinecolor="#444", color="white", showgrid=True)
fig.update_xaxes(**axis_style)
fig.update_yaxes(**axis_style)

fig.update_xaxes(title_text="Language", row=1, col=1)
fig.update_yaxes(title_text="Number of Films", row=1, col=1)
fig.update_xaxes(title_text="Proxy Score", row=1, col=2)
fig.update_yaxes(title_text="Revenue (₹ Crore)", row=1, col=2)

fig.update_xaxes(title_text="Year", row=2, col=1)
fig.update_yaxes(title_text="Female Cast Ratio", row=2, col=1)
fig.update_xaxes(title_text="Budget (₹ Crore)", row=2, col=2)
fig.update_yaxes(title_text="Revenue (₹ Crore)", row=2, col=2)

fig.update_xaxes(title_text="Year", row=3, col=1)
fig.update_yaxes(title_text="% of Films", row=3, col=1)
fig.update_xaxes(title_text="Revenue (₹ Crore)", row=3, col=2)

# Update subplot title text too
for annotation in fig.layout.annotations:
    annotation.font.size = 12
    annotation.font.color = "white"
    if "USD" in annotation.text:
        annotation.text = annotation.text.replace("USD Millions", "₹ Crore").replace("USD M", "₹ Crore")

fig.write_html("../visuals/dashboard.html",
               config={"displayModeBar": True, "responsive": True})
print("✓ Dashboard saved — open visuals/dashboard.html")
fig.show()

✓ Dashboard saved — open visuals/dashboard.html


In [14]:
# Add this as a new cell in 04_dashboard.ipynb or run standalone
import pandas as pd

df = pd.read_csv("./data/processed/indian_films_cleaned.csv")

# Laapataa Ladies actual box office: ~₹25 crore = ~$3M USD
df.loc[df["title"] == "Lost Ladies", "revenue_clean"] = 3000000
df.loc[df["title"] == "Lost Ladies", "revenue_m"] = 3.0

df.to_csv("./data/processed/indian_films_cleaned.csv", index=False)
print("✓ Fixed Lost Ladies revenue: $330M → $3M")

# Verify top 5 now
bo = df[df["has_boxoffice"]].copy()
bo["revenue_m"] = bo["revenue_clean"] / 1e6
print(bo.nlargest(5, "revenue_m")[["title", "revenue_m", "bechdel_proxy_score"]])

✓ Fixed Lost Ladies revenue: $330M → $3M
                          title  revenue_m  bechdel_proxy_score
2                        Dangal      311.0                  2.0
289  Bāhubali 2: The Conclusion      280.7                  0.0
285         Pushpa 2 - The Rule      219.0                  0.0
27     Zindagi Na Milegi Dobara      160.0                  3.0
284                         RRR      160.0                  2.0


In [25]:
fig.update_layout(
    height=1400,
    width=1400,
    title=dict(
        text="<b>Gender Representation in Indian Cinema (2000–2024)</b><br>"
             "<sup>Bechdel Proxy Analysis — Hindi, Tamil & Telugu Films</sup>",
        x=0.5, xanchor="center",
        font=dict(size=18, color="white")
    ),
    paper_bgcolor="#0f0f0f",
    plot_bgcolor="#1a1a1a",
    font=dict(color="white", family="Inter, Arial, sans-serif", size=11),
    barmode="group",
    legend=dict(
        bgcolor="#1a1a1a", bordercolor="#444", borderwidth=1,
        font=dict(color="white", size=10),
        orientation="v",
        x=1.01, y=0.99,
        xanchor="left", yanchor="top",
        tracegroupgap=4
    ),
    margin=dict(l=80, r=250, t=140, b=80)
)

# Per-axis styling
axis_style = dict(gridcolor="#2a2a2a", zerolinecolor="#444", color="white", showgrid=True)
fig.update_xaxes(**axis_style)
fig.update_yaxes(**axis_style)

# Row 1
fig.update_xaxes(title_text="Language", row=1, col=1)
fig.update_yaxes(title_text="Number of Films", row=1, col=1)
fig.update_xaxes(title_text="Proxy Score", row=1, col=2)
fig.update_yaxes(title_text="Revenue (₹ Crore)", row=1, col=2)

# Row 2
fig.update_xaxes(title_text="Year", row=2, col=1)
fig.update_yaxes(title_text="Female Cast Ratio", row=2, col=1)
fig.update_xaxes(title_text="Budget (₹ Crore)", row=2, col=2)
fig.update_yaxes(title_text="Revenue (₹ Crore)", row=2, col=2)

# Row 3
fig.update_xaxes(title_text="Year", row=3, col=1)
fig.update_yaxes(title_text="% of Films", row=3, col=1)
fig.update_xaxes(title_text="Revenue (₹ Crore)", row=3, col=2)

# Fix subplot title font
for annotation in fig.layout.annotations:
    annotation.font.size = 12
    annotation.font.color = "white"

# Save
fig.write_html("../visuals/dashboard.html", 
               config={"displayModeBar": True, "responsive": True})
print("✓ Dashboard saved — open visuals/dashboard.html")
fig.show()

✓ Dashboard saved — open visuals/dashboard.html


In [23]:
bo = df[df["has_boxoffice"]].copy()
bo["rev_cr"] = bo["revenue_inr_crore"]
bo["budget_cr"] = bo["budget_inr_crore"]

print(bo[bo["title"] == "Lost Ladies"][["title", "revenue_inr_crore", "budget_inr_crore", "bechdel_proxy_score", "year"]])
print("\nTop 5 by revenue (₹ crore):")
print(bo.nlargest(5, "rev_cr")[["title", "rev_cr", "budget_cr", "bechdel_proxy_score"]])

           title  revenue_inr_crore  budget_inr_crore  bechdel_proxy_score  \
144  Lost Ladies              25.26               NaN                  3.0   

     year  
144  2024  

Top 5 by revenue (₹ crore):
                          title   rev_cr  budget_cr  bechdel_proxy_score
2                        Dangal  2023.71    86.8400                  2.0
289  Bāhubali 2: The Conclusion  1810.00   258.8500                  0.0
285         Pushpa 2 - The Rule  1755.40   441.7150                  0.0
284                         RRR  1200.00   576.1500                  2.0
4                         Jawan  1148.00   301.8525                  2.0
